[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/optimization/currency.ipynb)

In [24]:
if 'google.colab' in str(get_ipython()):
    import os
    os.system('pip -qqq install ModelFlowIb')
    os.system('curl -L -o exchangerates_get.py https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/optimization/exchangerates_get.py')

In [25]:
import pandas as pd
import numpy as np

import exchangerates_get as er

# Flip to True to force a fresh ECB download and overwrite the cache.
REFRESH = False

In [26]:
fx_eur = er.ecb_fx_eur(
    currencies=["USD", "GBP", "JPY", "CHF","EUR","ZAR"],
    start="2010-01-01",
    freq='Q',
    refresh=REFRESH,
)
#fx_eur    

In [27]:
# Step 2: express everything in base currency 
fx_ccy = er.convert_base_currency(fx_eur, base="zar")

fx_returns = er.get_fx_returns(fx_ccy)

# Step 3: annualised covariance matrix
# ECB data is fetched at quarterly frequency (freq='Q') → periods_per_year=4
# annual_cov  =  4 × quarterly_cov
# annual_std  =  √4 × quarterly_std  =  2 × quarterly_std
fx_cov = er.get_fx_covariance(fx_returns, periods_per_year=4)

TypeError: get_fx_covariance() got an unexpected keyword argument 'periods_per_year'

In [ ]:
fx_ccy

In [ ]:
import numpy as np

fx_corr = er.get_fx_covariance(fx_returns, correlation=True)

# fx_cov is annualised (quarterly × 4) → std here is annual std
std = np.sqrt(np.diag(fx_cov.values))
std = pd.Series(std * 100, index=fx_cov.index, name="Annual std (%)")
print("Annual std (%) — note: quarterly std × 2 = annual std:")
print(std.to_string())
print()
print("Annual covariance matrix:")
display(fx_cov)

In [ ]:
print(std*100)

er.plot_corr_with_std(
    fx_returns,
    title="FX correlation (std on diagonal, %)"
)


er.plot_return_scatter_matrix_with_marginals(
    fx_returns,
    title="FX return scatterplot matrix (native frequency)"
)
#%%
er.plot_indexed_fx(
    fx_ccy,
    min_label_gap=10.0

)

In [ ]:
# cov_df is already annualised (quarterly × 4): risk axis is in annual units,
# consistent with the annual interest rates in the assumptions below.
cov_df = fx_cov.rename(
    index=lambda x: x.split('_')[1],
    columns=lambda x: x.split('_')[1],
)
names = cov_df.index

assumptions = pd.DataFrame(
    {
        'interest_rate':         [0.010, 0.020, 0.023, 0.013, 0.034],
        'expected_appreciation': [0.000, 0.000, 0.000, 0.000, 0.000],
        'min_share':             [0.0, 0.0, 0.0, 0.0, 0.0],
        'max_share':             [1.0, 1.0, 1.0, 1.0, 1.0],
        'current_share':         [0.2, 0.2, 0.2, 0.2, 0.2],
    },
    index=names,
)

res = er.mv_from_dataframes(cov_df=cov_df, assumptions=assumptions)

# row 0 = current composition; rows 1: = frontier
expected_cost = assumptions['interest_rate'] + assumptions['expected_appreciation']

er.plot_debt_frontier_labeled(
    res.iloc[1:].reset_index(drop=True),
    label_pos="start",
    cost_col="return",
    cost_s=expected_cost,
    cov_df=cov_df,
    current=res.iloc[0],
    export_path="zar_debt_frontier",
    export_formats=("png", "pdf", "svg"),
)

In [ ]:
res


## Interactive inputs — `DebtFrontierInputs`

The cells below wrap the same covariance matrix and assumption frame in a `DebtFrontierInputs` dataclass and render an editable grid (currencies × input fields). Edit any cell, then press **Run frontier** to re-solve and re-plot. The dataclass's `assumptions` attribute always reflects the latest edits, so you can also drive `inputs.plot()` from code between clicks.

In [ ]:
inputs = er.DebtFrontierInputs(cov_df=cov_df, assumptions=assumptions)
inputs.assumptions

In [ ]:
inputs.widget()